# ETN 5000 iterations: обработка результатов

Этот notebook обрабатывает все CSV из `all_etn_5000/results`:

- читает каждый CSV и нормализует названия метрик;
- считает среднее, median, std, min/max по ансамблю из 5 запусков;
- строит сравнение каждого метода с baseline `native BFGS` отдельно для AgPd/PdAg и MoNbTaVW;
- строит графики validation EPA RMSE, validation forces RMSE, train time, качество-время и истории обучения;
- сохраняет табличный отчет в `all_results/all_text_results_5000_iters.md`;
- сохраняет графики в `all_plots_images/5000_iters`.

Все пути задаются относительно корня `Testing-optimization-methods-in-MLP`. Исходные CSV не изменяются.


## 1. Импорты, стиль и вспомогательные функции

In [1]:

from pathlib import Path
import json
import ast
import re
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root(start=None):
    """Find Testing-optimization-methods-in-MLP root from current working directory."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "all_etn_5000" / "results").exists() and (candidate / "all_results").exists():
            return candidate
        nested = candidate / "Testing-optimization-methods-in-MLP"
        if (nested / "all_etn_5000" / "results").exists() and (nested / "all_results").exists():
            return nested
    raise RuntimeError("Cannot find Testing-optimization-methods-in-MLP project root")

PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "all_etn_5000" / "results"
TEXT_REPORT = PROJECT_ROOT / "all_results" / "all_text_results_5000_iters.md"
PLOT_DIR = PROJECT_ROOT / "all_plots_images" / "5000_iters"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_REPORT.parent.mkdir(parents=True, exist_ok=True)

sns.set_theme(
    context="talk",
    style="whitegrid",
    font="DejaVu Sans",
    rc={
        "figure.dpi": 140,
        "savefig.dpi": 220,
        "axes.edgecolor": "#9aa4ad",
        "axes.linewidth": 1.1,
        "grid.color": "#d9dee3",
        "grid.linewidth": 0.8,
        "axes.titleweight": "bold",
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "legend.title_fontsize": 12,
    },
)

BASELINE = "native BFGS"
BRIGHT = {
    "native BFGS": "#1f77b4",
    "SciPy BFGS": "#d62728",
    "SGD": "#ff7f0e",
    "Momentum": "#9467bd",
    "Adam": "#2ca02c",
    "Muon pad sqrt": "#17becf",
    "Muon factorization": "#8c564b",
}
ORDER = ["native BFGS", "SciPy BFGS", "Adam", "Momentum", "SGD", "Muon pad sqrt", "Muon factorization"]

METRIC_LABELS = {
    "train_epa_rmse": "train EPA RMSE (eV/atom)",
    "val_epa_rmse": "validation EPA RMSE (eV/atom)",
    "train_forces_rmse": "train forces RMSE (eV/A)",
    "val_forces_rmse": "validation forces RMSE (eV/A)",
    "train_time": "train time (seconds)",
    "final_loss": "final loss (objective units)",
    "steps": "steps",
    "epochs": "epochs",
    "nit": "SciPy iterations (nit)",
    "success": "success flag",
}

HISTORY_LABELS = {
    "losses": "training loss (objective units)",
    "grad_norms": "gradient norm (a.u.)",
    "lrs": "learning rate",
}


def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"Cannot read {path}: {exc}")
        return None


def parse_filename(path):
    name = Path(path).stem
    if name.startswith("AgPd_"):
        dataset = "AgPd/PdAg"
        rest = name.removeprefix("AgPd_results_etn_")
    elif name.startswith("MoNbTaVW_"):
        dataset = "MoNbTaVW"
        rest = name.removeprefix("MoNbTaVW_results_etn_")
    else:
        dataset = "unknown"
        rest = name

    mapping = {
        "native_bfgs": "native BFGS",
        "scipy_bfgs": "SciPy BFGS",
        "my_sgd": "SGD",
        "my_momentum": "Momentum",
        "my_adam": "Adam",
        "my_muon_pad_sqrt": "Muon pad sqrt",
        "my_muon_factorization": "Muon factorization",
    }
    optimizer = mapping.get(rest, rest)
    return dataset, optimizer


def normalize_columns(df):
    df = df.copy()
    rename = {
        "train_energy_atom_rmse": "train_epa_rmse",
        "val_energy_atom_rmse": "val_epa_rmse",
    }
    df = df.rename(columns=rename)
    return df


def load_one(path):
    df = safe_read_csv(path)
    if df is None:
        return None
    dataset, optimizer = parse_filename(path)
    df = normalize_columns(df)
    df["dataset"] = dataset
    df["optimizer"] = optimizer
    df["source"] = str(Path(path).relative_to(PROJECT_ROOT))
    return df


def parse_history(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if not isinstance(value, str) or not value.strip():
        return []
    text = value.strip()
    try:
        parsed = json.loads(text)
    except Exception:
        try:
            parsed = ast.literal_eval(text)
        except Exception:
            return []
    if not isinstance(parsed, (list, tuple)):
        return []
    out = []
    for item in parsed:
        try:
            x = float(item)
            if np.isfinite(x):
                out.append(x)
        except Exception:
            pass
    return out


def numeric_metric_columns(df):
    excluded = {"pot_num"}
    history = {"losses", "grad_norms", "lrs"}
    cols = []
    for col in df.columns:
        if col in excluded or col in history:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    return cols


def ensemble_summary(df):
    rows = []
    for metric in numeric_metric_columns(df):
        s = pd.to_numeric(df[metric], errors="coerce").dropna()
        if s.empty:
            continue
        rows.append({
            "dataset": df["dataset"].iloc[0],
            "optimizer": df["optimizer"].iloc[0],
            "metric": metric,
            "unit": metric_unit(metric),
            "n": int(s.shape[0]),
            "mean": s.mean(),
            "median": s.median(),
            "std": s.std(ddof=1) if s.shape[0] > 1 else 0.0,
                        "min": s.min(),
            "max": s.max(),
        })
    return pd.DataFrame(rows)


def wide_summary(summary):
    if summary.empty:
        return summary
    rows = []
    for (dataset, optimizer), g in summary.groupby(["dataset", "optimizer"], sort=False):
        row = {"dataset": dataset, "optimizer": optimizer}
        for _, r in g.iterrows():
            metric = r["metric"]
            row[f"{metric}_mean"] = r["mean"]
            row[f"{metric}_median"] = r["median"]
            row[f"{metric}_std"] = r["std"]
            row[f"{metric}_min"] = r["min"]
            row[f"{metric}_max"] = r["max"]
        rows.append(row)
    return pd.DataFrame(rows)


def comparison_vs_baseline(wide):
    rows = []
    all_metrics = sorted({col[:-5] for col in wide.columns if col.endswith("_mean")})
    priority = [
        "train_energy_rmse", "train_epa_rmse", "train_forces_rmse",
        "val_energy_rmse", "val_epa_rmse", "val_forces_rmse",
        "train_time", "steps", "epochs", "final_loss", "nit", "success",
    ]
    key_metrics = [m for m in priority if m in all_metrics] + [m for m in all_metrics if m not in priority]
    for dataset, g in wide.groupby("dataset", sort=False):
        base = g[g["optimizer"] == BASELINE]
        if base.empty:
            continue
        base = base.iloc[0]
        for _, row in g.iterrows():
            out = {"dataset": dataset, "optimizer": row["optimizer"], "baseline": BASELINE}
            for metric in key_metrics:
                mcol = f"{metric}_mean"
                scol = f"{metric}_std"
                if mcol not in g.columns or pd.isna(row.get(mcol, np.nan)):
                    continue
                val = row[mcol]
                bval = base.get(mcol, np.nan)
                out[f"{metric}_mean"] = val
                if scol in g.columns:
                    out[f"{metric}_std"] = row.get(scol, np.nan)
                if pd.notna(bval) and bval != 0:
                    out[f"{metric}_delta_vs_baseline"] = val - bval
                    out[f"{metric}_ratio_vs_baseline"] = val / bval
                    out[f"{metric}_pct_vs_baseline"] = (val / bval - 1.0) * 100.0
            rows.append(out)
    return pd.DataFrame(rows)


def metric_unit(metric):
    if "epa_rmse" in metric:
        return "eV/atom"
    if "forces_rmse" in metric:
        return "eV/A"
    if metric == "train_time":
        return "seconds"
    if metric in {"steps", "epochs", "nit"}:
        return "count"
    if metric == "success":
        return "flag"
    if metric == "final_loss":
        return "objective units"
    if "energy_rmse" in metric:
        return "eV"
    return ""


def nice_metric(metric):
    return METRIC_LABELS.get(metric, metric.replace("_", " "))


def slug(text):
    text = text.lower().replace("/", "_")
    text = re.sub(r"[^a-z0-9а-яё]+", "_", text, flags=re.IGNORECASE)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def optimizer_order(values):
    values = list(values)
    ordered = [x for x in ORDER if x in values]
    ordered += [x for x in values if x not in ordered]
    return ordered


def savefig(fig, filename, title, description, source):
    path = PLOT_DIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    PLOT_RECORDS.append({
        "filename": filename,
        "title": title,
        "description": description,
        "source": source,
    })
    return path


def plot_metric_bars(df, dataset):
    metrics = ["val_epa_rmse", "val_forces_rmse"]
    available = [m for m in metrics if m in df.columns]
    if not available:
        return
    order = optimizer_order(df["optimizer"].unique())
    fig, axes = plt.subplots(1, len(available), figsize=(7.5 * len(available), 5.8), squeeze=False)
    for ax, metric in zip(axes.flat, available):
        sns.barplot(
            data=df,
            x="optimizer",
            y=metric,
            order=order,
            palette=BRIGHT,
            errorbar="sd",
            capsize=0.18,
            err_kws={"linewidth": 1.2},
            ax=ax,
        )
        sns.stripplot(
            data=df,
            x="optimizer",
            y=metric,
            order=order,
            color="#222222",
            size=4,
            alpha=0.55,
            jitter=0.18,
            ax=ax,
        )
        base = df.loc[df["optimizer"] == BASELINE, metric].dropna()
        if not base.empty:
            ax.axhline(base.mean(), color="#1f77b4", linestyle="--", linewidth=1.5, alpha=0.8, label="native BFGS baseline")
        ax.set_title(nice_metric(metric))
        ax.set_xlabel("optimizer")
        ax.set_ylabel(nice_metric(metric))
        ax.tick_params(axis="x", rotation=30)
        if ax.get_legend() is not None:
            ax.legend(loc="upper right", frameon=True)
    fig.suptitle(f"{dataset}: validation errors, 5000 iterations", y=1.03, fontsize=20, fontweight="bold")
    fig.tight_layout()
    savefig(
        fig,
        f"{slug(dataset)}_validation_errors_5000.png",
        f"{dataset}: validation errors, 5000 iterations",
        "Mean+SD bars and individual ensemble runs for validation EPA and forces RMSE.",
        "all_etn_5000/results/*.csv",
    )


def plot_train_time_and_quality(df, dataset):
    if "train_time" not in df.columns:
        return
    order = optimizer_order(df["optimizer"].unique())
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8))
    sns.barplot(
        data=df,
        x="optimizer",
        y="train_time",
        order=order,
        palette=BRIGHT,
        errorbar="sd",
        capsize=0.18,
        err_kws={"linewidth": 1.2},
        ax=axes[0],
    )
    sns.stripplot(data=df, x="optimizer", y="train_time", order=order, color="#222222", size=4, alpha=0.5, jitter=0.18, ax=axes[0])
    axes[0].set_title("Train time")
    axes[0].set_xlabel("optimizer")
    axes[0].set_ylabel("train time (seconds)")
    axes[0].tick_params(axis="x", rotation=30)

    if "val_epa_rmse" in df.columns:
        sns.scatterplot(
            data=df,
            x="train_time",
            y="val_epa_rmse",
            hue="optimizer",
            style="optimizer",
            hue_order=order,
            palette=BRIGHT,
            s=90,
            edgecolor="white",
            linewidth=0.8,
            ax=axes[1],
        )
        axes[1].set_yscale("log")
        axes[1].set_title("Quality vs train time")
        axes[1].set_xlabel("train time (seconds)")
        axes[1].set_ylabel("validation EPA RMSE (eV/atom), log scale")
        axes[1].legend(title="optimizer", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    else:
        axes[1].axis("off")
    fig.suptitle(f"{dataset}: train time and quality, 5000 iterations", y=1.03, fontsize=20, fontweight="bold")
    fig.tight_layout()
    savefig(
        fig,
        f"{slug(dataset)}_train_time_quality_5000.png",
        f"{dataset}: train time and quality, 5000 iterations",
        "Train time with ensemble spread and validation EPA RMSE versus train time.",
        "all_etn_5000/results/*.csv",
    )


def plot_ensemble_spread(df, dataset):
    metrics = ["val_epa_rmse", "val_forces_rmse"]
    available = [m for m in metrics if m in df.columns]
    if not available:
        return
    order = optimizer_order(df["optimizer"].unique())
    fig, axes = plt.subplots(1, len(available), figsize=(7.5 * len(available), 5.8), squeeze=False)
    for ax, metric in zip(axes.flat, available):
        sns.barplot(
            data=df,
            x="optimizer",
            y=metric,
            order=order,
            palette=BRIGHT,
            errorbar="sd",
            capsize=0.18,
            err_kws={"linewidth": 1.2},
            ax=ax,
        )
        sns.stripplot(
            data=df,
            x="optimizer",
            y=metric,
            order=order,
            color="#111111",
            alpha=0.65,
            jitter=0.18,
            size=5,
            ax=ax,
        )
        base = df.loc[df["optimizer"] == BASELINE, metric].dropna()
        if not base.empty:
            ax.axhline(base.mean(), color="#1f77b4", linestyle="--", linewidth=1.4, alpha=0.8)
        ax.set_title(nice_metric(metric))
        ax.set_xlabel("optimizer")
        ax.set_ylabel(nice_metric(metric))
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle(f"{dataset}: ensemble spread, 5000 iterations", y=1.03, fontsize=20, fontweight="bold")
    fig.tight_layout()
    savefig(
        fig,
        f"{slug(dataset)}_ensemble_spread_5000.png",
        f"{dataset}: ensemble spread, 5000 iterations",
        "Mean+SD bars with individual points show variation across five runs for energy and forces validation errors.",
        "all_etn_5000/results/*.csv",
    )


def history_long(df, history_col):
    if history_col not in df.columns:
        return pd.DataFrame()
    rows = []
    for _, row in df.iterrows():
        values = parse_history(row.get(history_col))
        if not values:
            continue
        for i, value in enumerate(values):
            rows.append({
                "dataset": row["dataset"],
                "optimizer": row["optimizer"],
                "pot_num": row.get("pot_num", np.nan),
                "iteration": i,
                "value": value,
                "history": history_col,
            })
    return pd.DataFrame(rows)


def plot_history_overview(df, dataset, history_col="losses"):
    hist = history_long(df, history_col)
    if hist.empty:
        return
    order = optimizer_order(hist["optimizer"].unique())
    fig, ax = plt.subplots(figsize=(13, 6.2))
    for opt in order:
        sub = hist[hist["optimizer"] == opt]
        if sub.empty:
            continue
        color = BRIGHT.get(opt, None)
        for _, run in sub.groupby("pot_num"):
            ax.plot(run["iteration"], run["value"], color=color, alpha=0.12, linewidth=0.8)
        stat = sub.groupby("iteration", as_index=False)["value"].median()
        ax.plot(stat["iteration"], stat["value"], color=color, linewidth=2.4, label=opt)
    if history_col in {"losses", "grad_norms"}:
        positive = hist[hist["value"] > 0]
        if not positive.empty:
            ax.set_yscale("log")
    ax.set_title(f"{dataset}: {history_col.replace('_', ' ')} histories")
    ax.set_xlabel("iteration")
    ylabel = HISTORY_LABELS.get(history_col, history_col)
    if ax.get_yscale() == "log":
        ylabel += ", log scale"
    ax.set_ylabel(ylabel)
    ax.legend(title="optimizer", bbox_to_anchor=(-0.02, 1), loc="upper right", frameon=True)
    fig.tight_layout()
    savefig(
        fig,
        f"{slug(dataset)}_{history_col}_overview_5000.png",
        f"{dataset}: {history_col} histories, 5000 iterations",
        "Thin lines are individual runs; bold line is ensemble median by iteration.",
        "CSV history columns in all_etn_5000/results",
    )


def plot_history_individual(df, dataset, history_col="losses"):
    hist = history_long(df, history_col)
    if hist.empty:
        return
    for opt, sub in hist.groupby("optimizer", sort=False):
        fig, ax = plt.subplots(figsize=(11, 5.8))
        color = BRIGHT.get(opt, "#4c78a8")
        for _, run in sub.groupby("pot_num"):
            ax.plot(run["iteration"], run["value"], color=color, alpha=0.22, linewidth=0.9)
        stat = sub.groupby("iteration", as_index=False)["value"].median()
        ax.plot(stat["iteration"], stat["value"], color=color, linewidth=2.8, label=f"{opt} median")
        if history_col in {"losses", "grad_norms"}:
            positive = sub[sub["value"] > 0]
            if not positive.empty:
                ax.set_yscale("log")
        ax.set_title(f"{dataset}: {opt} {history_col.replace('_', ' ')}")
        ax.set_xlabel("iteration")
        ylabel = HISTORY_LABELS.get(history_col, history_col)
        if ax.get_yscale() == "log":
            ylabel += ", log scale"
        ax.set_ylabel(ylabel)
        ax.legend(loc="upper right", frameon=True)
        fig.tight_layout()
        savefig(
            fig,
            f"{slug(dataset)}_{slug(opt)}_{history_col}_5000.png",
            f"{dataset}: {opt} {history_col}, 5000 iterations",
            "Individual run histories plus median curve for one optimizer.",
            "CSV history columns in all_etn_5000/results",
        )



def plot_history_mean_band(df, dataset, history_col="losses", window=101):
    """Save one smoothed ensemble history plot per optimizer.

    Bold line is the rolling-smoothed ensemble mean. The band shows rolling-smoothed
    mean +/- std across runs at the same iteration. This intentionally suppresses
    batch-level noise and shows the optimization direction.
    """
    hist = history_long(df, history_col)
    if hist.empty:
        return
    for opt, sub in hist.groupby("optimizer", sort=False):
        grouped = sub.groupby("iteration")["value"]
        stat = grouped.agg(mean="mean", std="std").reset_index()
        for col in ["mean", "std"]:
            stat[col] = stat[col].rolling(window=window, center=True, min_periods=1).mean()
        stat["lower_std"] = (stat["mean"] - stat["std"].fillna(0)).clip(lower=1e-12)
        stat["upper_std"] = stat["mean"] + stat["std"].fillna(0)
        color = BRIGHT.get(opt, "#4c78a8")

        fig, ax = plt.subplots(figsize=(11.5, 5.8))
        ax.fill_between(
            stat["iteration"].to_numpy(),
            stat["lower_std"].to_numpy(),
            stat["upper_std"].to_numpy(),
            color=color,
            alpha=0.18,
            linewidth=0,
            label="mean +/- std",
        )
        ax.plot(stat["iteration"], stat["mean"], color=color, linewidth=3.0, label="ensemble mean")
        if history_col in {"losses", "grad_norms"}:
            positive = sub[sub["value"] > 0]
            if not positive.empty:
                ax.set_yscale("log")
        ax.set_title(f"{dataset}: {opt} ensemble {history_col.replace('_', ' ')}")
        ax.set_xlabel("iteration")
        ylabel = HISTORY_LABELS.get(history_col, history_col)
        if ax.get_yscale() == "log":
            ylabel += ", log scale"
        ax.set_ylabel(ylabel)
        ax.legend(loc="upper right", frameon=True)
        fig.tight_layout()
        savefig(
            fig,
            f"{slug(dataset)}_{slug(opt)}_{history_col}_ensemble_band_5000.png",
            f"{dataset}: {opt} ensemble {history_col}, 5000 iterations",
            "Smoothed ensemble mean with mean +/- std band; emphasizes training direction over batch noise.",
            "CSV history columns in all_etn_5000/results",
        )


def round_for_report(df):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{x:.6g}")
    return out


def df_to_csv_block(df):
    if df is None or df.empty:
        return "_Нет данных._\n"
    return "```csv\n" + round_for_report(df).to_csv(index=False) + "```\n"


def write_markdown_report(csv_files, per_file_summaries, summary_long, summary_wide, baseline_cmp):
    lines = []
    lines.append("# ETN 5000 iterations: итоговые текстовые результаты\n")
    lines.append("Источник: `all_etn_5000/results/*.csv`. Истории обучения (`losses`, `grad_norms`, `lrs`) не копируются целиком в markdown, но используются для графиков.\n")
    lines.append("## Индекс результатов\n")
    lines.append("### Исходные CSV\n")
    index_rows = []
    for p in csv_files:
        dataset, optimizer = parse_filename(p)
        rel = Path(p).relative_to(PROJECT_ROOT)
        df = loaded_by_path[str(p)]
        index_rows.append({
            "path": str(rel),
            "dataset": dataset,
            "optimizer": optimizer,
            "rows": len(df),
            "has_losses": "losses" in df.columns,
            "has_grad_norms": "grad_norms" in df.columns,
            "has_lrs": "lrs" in df.columns,
        })
    lines.append(df_to_csv_block(pd.DataFrame(index_rows)))
    lines.append("## Сводная таблица по ансамблям\n")
    lines.append("Для каждой пары dataset/optimizer посчитаны mean, median, std, min и max по численным метрикам. Std отражает разброс по ансамблю из пяти запусков.\n")
    lines.append(df_to_csv_block(summary_wide))
    lines.append("## Сравнение с baseline native BFGS\n")
    lines.append("Для каждого датасета baseline - `native BFGS`. В таблице есть абсолютная разница, отношение к baseline и процентное изменение для доступных метрик.\n")
    lines.append(df_to_csv_block(baseline_cmp))
    lines.append("## Детальные summary по каждому CSV\n")
    for p in csv_files:
        rel = Path(p).relative_to(PROJECT_ROOT)
        dataset, optimizer = parse_filename(p)
        lines.append(f"### {dataset}: {optimizer}\n")
        lines.append(f"Источник: `{rel}`\n")
        lines.append("Описание: ETN, 5000 итераций; агрегирование по 5 запускам ансамбля.\n")
        lines.append(df_to_csv_block(per_file_summaries[str(p)]))
    lines.append("## Графики\n")
    plot_df = pd.DataFrame(PLOT_RECORDS)
    if not plot_df.empty:
        plot_df.insert(0, "path", plot_df["filename"].map(lambda x: f"all_plots_images/5000_iters/{x}"))
        lines.append(df_to_csv_block(plot_df[["path", "title", "description", "source"]]))
    else:
        lines.append("_Графики не были построены._\n")
    lines.append("## Проверка полноты\n")
    lines.append(f"- CSV-файлов найдено: {len(csv_files)}.\n")
    lines.append(f"- CSV-файлов обработано: {len(csv_files)}.\n")
    lines.append(f"- Датасетов найдено: {summary_wide['dataset'].nunique() if not summary_wide.empty else 0}.\n")
    lines.append(f"- Методов оптимизации найдено: {summary_wide['optimizer'].nunique() if not summary_wide.empty else 0}.\n")
    lines.append(f"- Графиков сохранено: {len(PLOT_RECORDS)}.\n")
    missing_hist = []
    for p in csv_files:
        df = loaded_by_path[str(p)]
        if "losses" not in df.columns:
            missing_hist.append(str(Path(p).relative_to(PROJECT_ROOT)))
    lines.append("- CSV без history loss: " + (", ".join(f"`{x}`" for x in missing_hist) if missing_hist else "нет") + ".\n")
    lines.append("- Необработанные файлы: нет.\n")
    TEXT_REPORT.write_text("\n".join(lines), encoding="utf-8")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("TEXT_REPORT:", TEXT_REPORT)
print("PLOT_DIR:", PLOT_DIR)


PROJECT_ROOT: /Users/peterzarenkov/projects/course_work/Testing-optimization-methods-in-MLP
RESULTS_DIR: /Users/peterzarenkov/projects/course_work/Testing-optimization-methods-in-MLP/all_etn_5000/results
TEXT_REPORT: /Users/peterzarenkov/projects/course_work/Testing-optimization-methods-in-MLP/all_results/all_text_results_5000_iters.md
PLOT_DIR: /Users/peterzarenkov/projects/course_work/Testing-optimization-methods-in-MLP/all_plots_images/5000_iters


## 2. Чтение всех CSV

На этом шаге notebook находит все CSV в `all_etn_5000/results`, определяет dataset и optimizer из имени файла и приводит `energy_atom_rmse` к общему имени `epa_rmse`.

In [2]:

PLOT_RECORDS = []
csv_files = sorted(RESULTS_DIR.glob("*.csv"))
loaded = []
loaded_by_path = {}
for path in csv_files:
    df = load_one(path)
    if df is not None:
        loaded.append(df)
        loaded_by_path[str(path)] = df

if not loaded:
    raise RuntimeError(f"No CSV files found in {RESULTS_DIR}")

all_df = pd.concat(loaded, ignore_index=True, sort=False)
index = pd.DataFrame([
    {
        "path": str(path.relative_to(PROJECT_ROOT)),
        "dataset": parse_filename(path)[0],
        "optimizer": parse_filename(path)[1],
        "rows": len(loaded_by_path[str(path)]),
        "columns": ", ".join(loaded_by_path[str(path)].columns),
    }
    for path in csv_files
])
display(index)


,path,dataset,optimizer,rows,columns
0,all_etn_5000/results/AgPd_results_etn_my_adam.csv,AgPd/PdAg,Adam,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
1,all_etn_5000/results/AgPd_results_etn_my_momen...,AgPd/PdAg,Momentum,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
2,all_etn_5000/results/AgPd_results_etn_my_muon_...,AgPd/PdAg,Muon factorization,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
3,all_etn_5000/results/AgPd_results_etn_my_muon_...,AgPd/PdAg,Muon pad sqrt,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
4,all_etn_5000/results/AgPd_results_etn_my_sgd.csv,AgPd/PdAg,SGD,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
5,all_etn_5000/results/AgPd_results_etn_native_b...,AgPd/PdAg,native BFGS,5,"pot_num, train_energy_rmse, train_epa_rmse, tr..."
6,all_etn_5000/results/AgPd_results_etn_scipy_bf...,AgPd/PdAg,SciPy BFGS,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
7,all_etn_5000/results/MoNbTaVW_results_etn_my_a...,MoNbTaVW,Adam,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
8,all_etn_5000/results/MoNbTaVW_results_etn_my_m...,MoNbTaVW,Momentum,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."
9,all_etn_5000/results/MoNbTaVW_results_etn_my_m...,MoNbTaVW,Muon factorization,5,"pot_num, train_epa_rmse, train_forces_rmse, va..."


## 3. Сводная статистика по ансамблям

Для каждого метода считаются mean, median, std, min и max по всем численным метрикам. Std нужен как явная оценка разброса по ансамблю.

In [3]:

per_file_summaries = {}
summary_parts = []
for path in csv_files:
    df = loaded_by_path[str(path)]
    sm = ensemble_summary(df)
    per_file_summaries[str(path)] = sm
    summary_parts.append(sm)

summary_long = pd.concat(summary_parts, ignore_index=True)
summary_wide = wide_summary(summary_long)
summary_wide["optimizer"] = pd.Categorical(summary_wide["optimizer"], categories=ORDER, ordered=True)
summary_wide = summary_wide.sort_values(["dataset", "optimizer"]).reset_index(drop=True)
summary_wide["optimizer"] = summary_wide["optimizer"].astype(str)

display(summary_wide)


,dataset,optimizer,train_epa_rmse_mean,train_epa_rmse_median,train_epa_rmse_std,train_epa_rmse_min,train_epa_rmse_max,train_forces_rmse_mean,train_forces_rmse_median,train_forces_rmse_std,...,nit_mean,nit_median,nit_std,nit_min,nit_max,success_mean,success_median,success_std,success_min,success_max
0,AgPd/PdAg,native BFGS,0.008614,0.008614,3.025209e-13,0.008614,0.008614,0.191736,0.191736,1.246965e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AgPd/PdAg,SciPy BFGS,0.019703,0.008552,2.493359e-02,0.008552,0.064305,0.290148,0.191504,2.205765e-01,...,98.6,99.0,17.643696,75.0,117.0,0.4,0.0,0.547723,0.0,1.0
2,AgPd/PdAg,Adam,0.019873,0.019918,3.594094e-04,0.019376,0.020309,0.606935,0.606932,1.184780e-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AgPd/PdAg,Momentum,0.128271,0.128264,1.022384e-05,0.128262,0.128282,1.401374,1.401374,1.048852e-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AgPd/PdAg,SGD,0.128263,0.128263,2.200573e-06,0.128260,0.128266,1.401374,1.401374,1.046508e-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,AgPd/PdAg,Muon pad sqrt,0.019509,0.019738,1.113756e-03,0.017710,0.020742,0.639855,0.613511,4.954726e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,AgPd/PdAg,Muon factorization,0.028811,0.029053,5.261970e-04,0.028097,0.029275,1.091001,1.123435,6.371148e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,MoNbTaVW,native BFGS,0.417334,0.417334,2.615388e-10,0.417334,0.417334,0.515170,0.515170,8.090178e-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MoNbTaVW,SciPy BFGS,0.359549,0.359549,1.028979e-09,0.359549,0.359549,0.516251,0.516251,4.467417e-10,...,237.2,237.0,69.908512,143.0,305.0,0.0,0.0,0.000000,0.0,0.0
9,MoNbTaVW,Adam,0.711235,0.710717,9.308472e-04,0.710324,0.712472,0.829680,0.830166,1.761790e-03,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## AgPd/PdAg: Adam

Источник: `all_etn_5000/results/AgPd_results_etn_my_adam.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [4]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_my_adam.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,Adam,train_epa_rmse,eV/atom,5,0.019873,0.019918,0.000359,0.019376,0.020309
1,AgPd/PdAg,Adam,train_forces_rmse,eV/A,5,0.606935,0.606932,0.000118,0.606802,0.607086
2,AgPd/PdAg,Adam,val_epa_rmse,eV/atom,5,0.022994,0.022888,0.000650,0.022224,0.024029
3,AgPd/PdAg,Adam,val_forces_rmse,eV/A,5,0.880831,0.880802,0.000260,0.880599,0.881256
4,AgPd/PdAg,Adam,train_time,seconds,5,1068.606557,1061.247429,16.973553,1055.276335,1096.699591
5,AgPd/PdAg,Adam,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,AgPd/PdAg,Adam,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,AgPd/PdAg,Adam,final_loss,objective units,5,7.943223,8.708321,1.506833,5.454680,9.189080


## AgPd/PdAg: Momentum

Источник: `all_etn_5000/results/AgPd_results_etn_my_momentum.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [5]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_my_momentum.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,Momentum,train_epa_rmse,eV/atom,5,0.128271,0.128264,1.022384e-05,0.128262,0.128282
1,AgPd/PdAg,Momentum,train_forces_rmse,eV/A,5,1.401374,1.401374,1.048852e-09,1.401374,1.401374
2,AgPd/PdAg,Momentum,val_epa_rmse,eV/atom,5,0.172698,0.172596,1.642668e-03,0.170234,0.174764
3,AgPd/PdAg,Momentum,val_forces_rmse,eV/A,5,1.730164,1.730164,6.632085e-10,1.730164,1.730164
4,AgPd/PdAg,Momentum,train_time,seconds,5,1063.969828,1062.072549,4.438859e+00,1060.052411,1069.929674
5,AgPd/PdAg,Momentum,steps,count,5,5000.000000,5000.000000,0.000000e+00,5000.000000,5000.000000
6,AgPd/PdAg,Momentum,epochs,count,5,999.000000,999.000000,0.000000e+00,999.000000,999.000000
7,AgPd/PdAg,Momentum,final_loss,objective units,5,54.193740,56.351347,6.655999e+00,42.483799,59.038564


## AgPd/PdAg: Muon factorization

Источник: `all_etn_5000/results/AgPd_results_etn_my_muon_factorization.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [6]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_my_muon_factorization.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,Muon factorization,train_epa_rmse,eV/atom,5,0.028811,0.029053,0.000526,0.028097,0.029275
1,AgPd/PdAg,Muon factorization,train_forces_rmse,eV/A,5,1.091001,1.123435,0.063711,0.983140,1.134737
2,AgPd/PdAg,Muon factorization,val_epa_rmse,eV/atom,5,0.045087,0.044938,0.000667,0.044458,0.045846
3,AgPd/PdAg,Muon factorization,val_forces_rmse,eV/A,5,1.516583,1.554210,0.075377,1.389981,1.570454
4,AgPd/PdAg,Muon factorization,train_time,seconds,5,1063.455305,1062.913436,1.411292,1062.120988,1065.681207
5,AgPd/PdAg,Muon factorization,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,AgPd/PdAg,Muon factorization,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,AgPd/PdAg,Muon factorization,final_loss,objective units,5,27.715621,29.864074,6.147162,19.110878,33.500502


## AgPd/PdAg: Muon pad sqrt

Источник: `all_etn_5000/results/AgPd_results_etn_my_muon_pad_sqrt.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [7]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_my_muon_pad_sqrt.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,Muon pad sqrt,train_epa_rmse,eV/atom,5,0.019509,0.019738,0.001114,0.017710,0.020742
1,AgPd/PdAg,Muon pad sqrt,train_forces_rmse,eV/A,5,0.639855,0.613511,0.049547,0.610725,0.726090
2,AgPd/PdAg,Muon pad sqrt,val_epa_rmse,eV/atom,5,0.025005,0.023895,0.003325,0.022868,0.030888
3,AgPd/PdAg,Muon pad sqrt,val_forces_rmse,eV/A,5,0.936555,0.892694,0.079659,0.885176,1.071493
4,AgPd/PdAg,Muon pad sqrt,train_time,seconds,5,1064.180055,1062.387906,4.124307,1059.314590,1069.366780
5,AgPd/PdAg,Muon pad sqrt,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,AgPd/PdAg,Muon pad sqrt,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,AgPd/PdAg,Muon pad sqrt,final_loss,objective units,5,9.382184,9.462977,3.188856,5.490345,14.052763


## AgPd/PdAg: SGD

Источник: `all_etn_5000/results/AgPd_results_etn_my_sgd.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [8]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_my_sgd.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,SGD,train_epa_rmse,eV/atom,5,0.128263,0.128263,2.200573e-06,0.128260,0.128266
1,AgPd/PdAg,SGD,train_forces_rmse,eV/A,5,1.401374,1.401374,1.046508e-09,1.401374,1.401374
2,AgPd/PdAg,SGD,val_epa_rmse,eV/atom,5,0.172721,0.173026,1.135380e-03,0.171314,0.173991
3,AgPd/PdAg,SGD,val_forces_rmse,eV/A,5,1.730164,1.730164,6.618068e-10,1.730164,1.730164
4,AgPd/PdAg,SGD,train_time,seconds,5,1061.031991,1061.461493,1.362839e+00,1058.707729,1062.161639
5,AgPd/PdAg,SGD,steps,count,5,5000.000000,5000.000000,0.000000e+00,5000.000000,5000.000000
6,AgPd/PdAg,SGD,epochs,count,5,999.000000,999.000000,0.000000e+00,999.000000,999.000000
7,AgPd/PdAg,SGD,final_loss,objective units,5,54.305594,56.469047,6.584345e+00,42.712713,59.034809


## AgPd/PdAg: native BFGS

Источник: `all_etn_5000/results/AgPd_results_etn_native_bfgs.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [9]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_native_bfgs.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,native BFGS,train_energy_rmse,eV,5,0.275656,0.275656,9.681456e-12,0.275656,0.275656
1,AgPd/PdAg,native BFGS,train_epa_rmse,eV/atom,5,0.008614,0.008614,3.025209e-13,0.008614,0.008614
2,AgPd/PdAg,native BFGS,train_forces_rmse,eV/A,5,0.191736,0.191736,1.246965e-12,0.191736,0.191736
3,AgPd/PdAg,native BFGS,val_energy_rmse,eV,5,0.443984,0.443984,8.655737e-11,0.443984,0.443984
4,AgPd/PdAg,native BFGS,val_epa_rmse,eV/atom,5,0.013874,0.013874,2.704909e-12,0.013874,0.013874
5,AgPd/PdAg,native BFGS,val_forces_rmse,eV/A,5,0.277115,0.277115,9.102695e-12,0.277115,0.277115
6,AgPd/PdAg,native BFGS,train_time,seconds,5,60.505358,62.524552,1.266626e+01,45.118854,78.073981


## AgPd/PdAg: SciPy BFGS

Источник: `all_etn_5000/results/AgPd_results_etn_scipy_bfgs.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [10]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/AgPd_results_etn_scipy_bfgs.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,AgPd/PdAg,SciPy BFGS,train_epa_rmse,eV/atom,5,0.019703,0.008552,2.493359e-02,0.008552,0.064305
1,AgPd/PdAg,SciPy BFGS,train_forces_rmse,eV/A,5,0.290148,0.191504,2.205765e-01,0.191504,0.684728
2,AgPd/PdAg,SciPy BFGS,val_epa_rmse,eV/atom,5,0.033291,0.013971,4.320114e-02,0.013971,0.110571
3,AgPd/PdAg,SciPy BFGS,val_forces_rmse,eV/A,5,0.401276,0.277264,2.772991e-01,0.277264,0.897323
4,AgPd/PdAg,SciPy BFGS,train_time,seconds,5,81.472478,76.726029,2.379655e+01,56.727377,117.984697
5,AgPd/PdAg,SciPy BFGS,nit,count,5,98.600000,99.000000,1.764370e+01,75.000000,117.000000
6,AgPd/PdAg,SciPy BFGS,success,flag,5,0.400000,0.000000,5.477226e-01,0.000000,1.000000
7,AgPd/PdAg,SciPy BFGS,final_loss,objective units,5,5.832470,5.832470,1.191707e-13,5.832470,5.832470


## MoNbTaVW: Adam

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_my_adam.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [11]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_my_adam.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,Adam,train_epa_rmse,eV/atom,5,0.711235,0.710717,0.000931,0.710324,0.712472
1,MoNbTaVW,Adam,train_forces_rmse,eV/A,5,0.829680,0.830166,0.001762,0.826793,0.831307
2,MoNbTaVW,Adam,val_epa_rmse,eV/atom,5,0.833041,0.833523,0.001748,0.830278,0.834524
3,MoNbTaVW,Adam,val_forces_rmse,eV/A,5,0.832387,0.832796,0.001222,0.830356,0.833474
4,MoNbTaVW,Adam,train_time,seconds,5,2876.155792,2874.183623,16.283189,2855.697206,2900.699726
5,MoNbTaVW,Adam,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,MoNbTaVW,Adam,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,MoNbTaVW,Adam,final_loss,objective units,5,95.257154,87.053154,37.052328,50.158300,149.364479


## MoNbTaVW: Momentum

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_my_momentum.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [12]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_my_momentum.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,Momentum,train_epa_rmse,eV/atom,5,5.162607,5.162803,4.212116e-04,5.162082,5.163019
1,MoNbTaVW,Momentum,train_forces_rmse,eV/A,5,0.900020,0.900020,2.344897e-10,0.900020,0.900020
2,MoNbTaVW,Momentum,val_epa_rmse,eV/atom,5,5.198268,5.198530,4.391230e-04,5.197691,5.198650
3,MoNbTaVW,Momentum,val_forces_rmse,eV/A,5,0.887349,0.887349,2.405360e-10,0.887349,0.887349
4,MoNbTaVW,Momentum,train_time,seconds,5,2863.056640,2858.679502,1.264979e+01,2850.852424,2877.887657
5,MoNbTaVW,Momentum,steps,count,5,5000.000000,5000.000000,0.000000e+00,5000.000000,5000.000000
6,MoNbTaVW,Momentum,epochs,count,5,999.000000,999.000000,0.000000e+00,999.000000,999.000000
7,MoNbTaVW,Momentum,final_loss,objective units,5,40792.987425,42593.027245,5.696247e+03,31817.110403,46836.667476


## MoNbTaVW: Muon factorization

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_my_muon_factorization.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [13]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_my_muon_factorization.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,Muon factorization,train_epa_rmse,eV/atom,5,0.452674,0.455129,0.016438,0.435685,0.471308
1,MoNbTaVW,Muon factorization,train_forces_rmse,eV/A,5,0.742859,0.742191,0.002579,0.739659,0.745625
2,MoNbTaVW,Muon factorization,val_epa_rmse,eV/atom,5,0.515089,0.523411,0.019283,0.493103,0.535495
3,MoNbTaVW,Muon factorization,val_forces_rmse,eV/A,5,0.747439,0.746660,0.002465,0.744281,0.750106
4,MoNbTaVW,Muon factorization,train_time,seconds,5,2855.885384,2861.618589,17.033988,2837.609877,2874.252411
5,MoNbTaVW,Muon factorization,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,MoNbTaVW,Muon factorization,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,MoNbTaVW,Muon factorization,final_loss,objective units,5,68.875647,60.937427,24.416138,37.509433,100.545426


## MoNbTaVW: Muon pad sqrt

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_my_muon_pad_sqrt.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [14]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_my_muon_pad_sqrt.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,Muon pad sqrt,train_epa_rmse,eV/atom,5,0.452674,0.455129,0.016438,0.435685,0.471308
1,MoNbTaVW,Muon pad sqrt,train_forces_rmse,eV/A,5,0.742859,0.742191,0.002579,0.739659,0.745625
2,MoNbTaVW,Muon pad sqrt,val_epa_rmse,eV/atom,5,0.515089,0.523411,0.019283,0.493103,0.535495
3,MoNbTaVW,Muon pad sqrt,val_forces_rmse,eV/A,5,0.747439,0.746660,0.002465,0.744281,0.750106
4,MoNbTaVW,Muon pad sqrt,train_time,seconds,5,2863.088362,2856.513429,14.676553,2846.900672,2883.802528
5,MoNbTaVW,Muon pad sqrt,steps,count,5,5000.000000,5000.000000,0.000000,5000.000000,5000.000000
6,MoNbTaVW,Muon pad sqrt,epochs,count,5,999.000000,999.000000,0.000000,999.000000,999.000000
7,MoNbTaVW,Muon pad sqrt,final_loss,objective units,5,68.875647,60.937427,24.416138,37.509433,100.545426


## MoNbTaVW: SGD

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_my_sgd.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [15]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_my_sgd.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,SGD,train_epa_rmse,eV/atom,5,5.153295,5.153489,4.200347e-04,5.152769,5.153706
1,MoNbTaVW,SGD,train_forces_rmse,eV/A,5,0.900020,0.900020,2.344702e-10,0.900020,0.900020
2,MoNbTaVW,SGD,val_epa_rmse,eV/atom,5,5.189112,5.189373,4.382689e-04,5.188534,5.189495
3,MoNbTaVW,SGD,val_forces_rmse,eV/A,5,0.887349,0.887349,2.405184e-10,0.887349,0.887349
4,MoNbTaVW,SGD,train_time,seconds,5,2864.668913,2863.630371,1.671687e+01,2838.691426,2881.024809
5,MoNbTaVW,SGD,steps,count,5,5000.000000,5000.000000,0.000000e+00,5000.000000,5000.000000
6,MoNbTaVW,SGD,epochs,count,5,999.000000,999.000000,0.000000e+00,999.000000,999.000000
7,MoNbTaVW,SGD,final_loss,objective units,5,40644.862772,42439.108963,5.677658e+03,31698.864296,46668.136961


## MoNbTaVW: native BFGS

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_native_bfgs.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [16]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_native_bfgs.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,native BFGS,train_energy_rmse,eV,5,7.557570,7.557570,3.655226e-09,7.557570,7.557570
1,MoNbTaVW,native BFGS,train_epa_rmse,eV/atom,5,0.417334,0.417334,2.615388e-10,0.417334,0.417334
2,MoNbTaVW,native BFGS,train_forces_rmse,eV/A,5,0.515170,0.515170,8.090178e-11,0.515170,0.515170
3,MoNbTaVW,native BFGS,val_energy_rmse,eV,5,6.054119,6.054119,4.897484e-09,6.054119,6.054119
4,MoNbTaVW,native BFGS,val_epa_rmse,eV/atom,5,0.476876,0.476876,2.337587e-10,0.476876,0.476876
5,MoNbTaVW,native BFGS,val_forces_rmse,eV/A,5,0.523471,0.523471,8.204665e-11,0.523471,0.523471
6,MoNbTaVW,native BFGS,train_time,seconds,5,151.640761,140.759787,4.552984e+01,103.870045,226.269005


## MoNbTaVW: SciPy BFGS

Источник: `all_etn_5000/results/MoNbTaVW_results_etn_scipy_bfgs.csv`. Ниже summary по ансамблю из пяти запусков: mean, median, std, min/max.

In [17]:
display(per_file_summaries[str(PROJECT_ROOT / 'all_etn_5000/results/MoNbTaVW_results_etn_scipy_bfgs.csv')])

,dataset,optimizer,metric,unit,n,mean,median,std,min,max
0,MoNbTaVW,SciPy BFGS,train_epa_rmse,eV/atom,5,0.359549,0.359549,1.028979e-09,0.359549,0.359549
1,MoNbTaVW,SciPy BFGS,train_forces_rmse,eV/A,5,0.516251,0.516251,4.467417e-10,0.516251,0.516251
2,MoNbTaVW,SciPy BFGS,val_epa_rmse,eV/atom,5,0.499952,0.499952,4.275921e-10,0.499952,0.499952
3,MoNbTaVW,SciPy BFGS,val_forces_rmse,eV/A,5,0.529053,0.529053,1.301382e-09,0.529053,0.529053
4,MoNbTaVW,SciPy BFGS,train_time,seconds,5,251.846313,269.732638,7.293923e+01,158.009841,346.290136
5,MoNbTaVW,SciPy BFGS,nit,count,5,237.200000,237.000000,6.990851e+01,143.000000,305.000000
6,MoNbTaVW,SciPy BFGS,success,flag,5,0.000000,0.000000,0.000000e+00,0.000000,0.000000
7,MoNbTaVW,SciPy BFGS,final_loss,objective units,5,188.867473,188.867473,5.587595e-13,188.867473,188.867473


## 4. Сравнение всех методов с baseline native BFGS

Baseline считается отдельно для каждого датасета. Для доступных метрик выводятся mean/std, delta, ratio и процентное изменение относительно baseline.

In [18]:

baseline_cmp = comparison_vs_baseline(summary_wide)
if not baseline_cmp.empty:
    baseline_cmp["optimizer"] = pd.Categorical(baseline_cmp["optimizer"], categories=ORDER, ordered=True)
    baseline_cmp = baseline_cmp.sort_values(["dataset", "optimizer"]).reset_index(drop=True)
    baseline_cmp["optimizer"] = baseline_cmp["optimizer"].astype(str)

display(baseline_cmp)


,dataset,optimizer,baseline,train_energy_rmse_mean,train_energy_rmse_std,train_energy_rmse_delta_vs_baseline,train_energy_rmse_ratio_vs_baseline,train_energy_rmse_pct_vs_baseline,train_epa_rmse_mean,train_epa_rmse_std,...,final_loss_mean,final_loss_std,nit_mean,nit_std,success_mean,success_std,steps_mean,steps_std,epochs_mean,epochs_std
0,AgPd/PdAg,native BFGS,native BFGS,0.275656,9.681456e-12,0.0,1.0,0.0,0.008614,3.025209e-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AgPd/PdAg,SciPy BFGS,native BFGS,NaN,NaN,NaN,NaN,NaN,0.019703,2.493359e-02,...,5.832470,1.191707e-13,98.6,17.643696,0.4,0.547723,NaN,NaN,NaN,NaN
2,AgPd/PdAg,Adam,native BFGS,NaN,NaN,NaN,NaN,NaN,0.019873,3.594094e-04,...,7.943223,1.506833e+00,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0
3,AgPd/PdAg,Momentum,native BFGS,NaN,NaN,NaN,NaN,NaN,0.128271,1.022384e-05,...,54.193740,6.655999e+00,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0
4,AgPd/PdAg,SGD,native BFGS,NaN,NaN,NaN,NaN,NaN,0.128263,2.200573e-06,...,54.305594,6.584345e+00,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0
5,AgPd/PdAg,Muon pad sqrt,native BFGS,NaN,NaN,NaN,NaN,NaN,0.019509,1.113756e-03,...,9.382184,3.188856e+00,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0
6,AgPd/PdAg,Muon factorization,native BFGS,NaN,NaN,NaN,NaN,NaN,0.028811,5.261970e-04,...,27.715621,6.147162e+00,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0
7,MoNbTaVW,native BFGS,native BFGS,7.557570,3.655226e-09,0.0,1.0,0.0,0.417334,2.615388e-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MoNbTaVW,SciPy BFGS,native BFGS,NaN,NaN,NaN,NaN,NaN,0.359549,1.028979e-09,...,188.867473,5.587595e-13,237.2,69.908512,0.0,0.000000,NaN,NaN,NaN,NaN
9,MoNbTaVW,Adam,native BFGS,NaN,NaN,NaN,NaN,NaN,0.711235,9.308472e-04,...,95.257154,3.705233e+01,NaN,NaN,NaN,NaN,5000.0,0.0,999.0,0.0


## 5. Построение и сохранение графиков

Графики сохраняются в `all_plots_images/5000_iters`. На осях указаны единицы измерения: eV/atom для EPA RMSE, eV/A для forces RMSE, seconds для времени.

In [19]:

for dataset, df in all_df.groupby("dataset", sort=False):
    display(Markdown(f"## Графики для {dataset}"))
    plot_metric_bars(df, dataset)
    plot_train_time_and_quality(df, dataset)
    plot_ensemble_spread(df, dataset)
    plot_history_overview(df, dataset, "losses")
    plot_history_individual(df, dataset, "losses")
    plot_history_mean_band(df, dataset, "losses", window=101)
    # Supplementary diagnostics are saved when available. They are useful for debugging optimizer behavior.
    plot_history_overview(df, dataset, "grad_norms")
    plot_history_overview(df, dataset, "lrs")

plot_manifest = pd.DataFrame(PLOT_RECORDS)
if not plot_manifest.empty:
    plot_manifest.to_csv(PLOT_DIR / "manifest_5000_iters.csv", index=False)
    captions = ["# ETN 5000 iterations: exported plots\n"]
    for i, row in plot_manifest.iterrows():
        captions.append(f"## {i + 1}. {row['title']}\n")
        captions.append(f"Файл: `{row['filename']}`\n")
        captions.append(f"Описание: {row['description']}\n")
        captions.append(f"Источник: `{row['source']}`\n")
    (PLOT_DIR / "captions_5000_iters.md").write_text("\n".join(captions), encoding="utf-8")

display(plot_manifest)


## Графики для AgPd/PdAg

## Графики для MoNbTaVW

,filename,title,description,source
0,agpd_pdag_validation_errors_5000.png,"AgPd/PdAg: validation errors, 5000 iterations",Mean+SD bars and individual ensemble runs for ...,all_etn_5000/results/*.csv
1,agpd_pdag_train_time_quality_5000.png,"AgPd/PdAg: train time and quality, 5000 iterat...",Train time with ensemble spread and validation...,all_etn_5000/results/*.csv
2,agpd_pdag_ensemble_spread_5000.png,"AgPd/PdAg: ensemble spread, 5000 iterations",Mean+SD bars with individual points show varia...,all_etn_5000/results/*.csv
3,agpd_pdag_losses_overview_5000.png,"AgPd/PdAg: losses histories, 5000 iterations",Thin lines are individual runs; bold line is e...,CSV history columns in all_etn_5000/results
4,agpd_pdag_adam_losses_5000.png,"AgPd/PdAg: Adam losses, 5000 iterations",Individual run histories plus median curve for...,CSV history columns in all_etn_5000/results
5,agpd_pdag_momentum_losses_5000.png,"AgPd/PdAg: Momentum losses, 5000 iterations",Individual run histories plus median curve for...,CSV history columns in all_etn_5000/results
6,agpd_pdag_muon_factorization_losses_5000.png,"AgPd/PdAg: Muon factorization losses, 5000 ite...",Individual run histories plus median curve for...,CSV history columns in all_etn_5000/results
7,agpd_pdag_muon_pad_sqrt_losses_5000.png,"AgPd/PdAg: Muon pad sqrt losses, 5000 iterations",Individual run histories plus median curve for...,CSV history columns in all_etn_5000/results
8,agpd_pdag_sgd_losses_5000.png,"AgPd/PdAg: SGD losses, 5000 iterations",Individual run histories plus median curve for...,CSV history columns in all_etn_5000/results
9,agpd_pdag_adam_losses_ensemble_band_5000.png,"AgPd/PdAg: Adam ensemble losses, 5000 iterations",Smoothed ensemble mean with mean +/- std band;...,CSV history columns in all_etn_5000/results


## 6. Экспорт текстового отчета

In [20]:

write_markdown_report(csv_files, per_file_summaries, summary_long, summary_wide, baseline_cmp)
print(f"Markdown report written to: {TEXT_REPORT.relative_to(PROJECT_ROOT)}")
print(f"Plots written to: {PLOT_DIR.relative_to(PROJECT_ROOT)}")
print(f"Plots saved: {len(PLOT_RECORDS)}")


Markdown report written to: all_results/all_text_results_5000_iters.md
Plots written to: all_plots_images/5000_iters
Plots saved: 32


## 7. Проверка полноты

In [21]:

expected_outputs = [TEXT_REPORT, PLOT_DIR / "manifest_5000_iters.csv", PLOT_DIR / "captions_5000_iters.md"]
for path in expected_outputs:
    print(path.relative_to(PROJECT_ROOT), "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)
print("PNG count:", len(list(PLOT_DIR.glob("*.png"))))


all_results/all_text_results_5000_iters.md exists= True size= 34690
all_plots_images/5000_iters/manifest_5000_iters.csv exists= True size= 6760
all_plots_images/5000_iters/captions_5000_iters.md exists= True size= 8643
PNG count: 32
